# Project 2: Quantum Key Distribution — BB84 Protocol

---

## Overview

Quantum Key Distribution (QKD) enables two parties — Alice and Bob — to establish a shared secret key with **information-theoretic security** guaranteed by the laws of quantum mechanics. Unlike classical key exchange (RSA, Diffie-Hellman), QKD is secure against **any** computational attack, including quantum computers.

The **BB84 protocol**, proposed by Charles Bennett and Gilles Brassard in 1984, was the first QKD protocol and remains the most widely implemented.

### What We'll Build

1. Full BB84 protocol simulation (Alice, quantum channel, Bob)
2. Eavesdropper (Eve) detection via Quantum Bit Error Rate (QBER)
3. Information reconciliation and privacy amplification
4. Security analysis with visualizations
5. Comparison with the simplified B92 protocol

### Key Principles

| Principle | Role in QKD |
|-----------|-------------|
| **No-Cloning Theorem** | Eve cannot copy qubits without disturbing them |
| **Heisenberg Uncertainty** | Measuring in wrong basis destroys information |
| **Quantum Superposition** | Enables encoding in conjugate bases |

In [ ]:
# Imports
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

np.random.seed(42)
random.seed(42)

print(f"PennyLane version: {qml.__version__}")
print("Project 2: Quantum Key Distribution - BB84 Protocol")

## 1. Theoretical Foundation

### No-Cloning Theorem

There exists no unitary operation $U$ such that for arbitrary quantum states $|\psi\rangle$:

$$U|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle$$

**Proof sketch**: Suppose $U|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle$ and $U|\phi\rangle|0\rangle = |\phi\rangle|\phi\rangle$. Taking the inner product:

$$\langle\psi|\phi\rangle = (\langle\psi|\phi\rangle)^2$$

This is only satisfied when $\langle\psi|\phi\rangle = 0$ or $1$, meaning cloning only works for orthogonal or identical states.

### Conjugate Bases

BB84 uses two **mutually unbiased bases**:

- **Z-basis (rectilinear)**: $\{|0\rangle, |1\rangle\}$
- **X-basis (diagonal)**: $\{|+\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}, \;|-\rangle = \frac{|0\rangle-|1\rangle}{\sqrt{2}}\}$

Measuring a Z-basis state in the X-basis (or vice versa) gives a **completely random** outcome:

$$|\langle +|0\rangle|^2 = |\langle +|1\rangle|^2 = \frac{1}{2}$$

### BB84 Encoding Table

| Bit | Z-basis | X-basis |
|-----|---------|--------|
| 0 | $|0\rangle$ | $|+\rangle$ |
| 1 | $|1\rangle$ | $|-\rangle$ |

## 2. BB84 Protocol Steps

1. **Alice** generates random bits and random bases, prepares qubits accordingly
2. **Alice** sends qubits to **Bob** through a quantum channel
3. **Bob** randomly chooses measurement bases and measures each qubit
4. **Basis reconciliation** (classical channel): Alice and Bob publicly compare bases
5. **Key sifting**: Keep only bits where bases matched (~50% of bits)
6. **Error estimation**: Compare a subset to check for eavesdropping
7. **Privacy amplification**: Shorten key to remove any leaked information

In [ ]:
# Full BB84 Protocol Simulation

dev = qml.device("default.qubit", wires=1, shots=1)

@qml.qnode(dev)
def bb84_prepare_and_measure(alice_bit, alice_basis, bob_basis):
    """Simulate BB84: Alice prepares, Bob measures."""
    # Alice's preparation
    if alice_bit == 1:
        qml.PauliX(wires=0)  # Flip to |1>
    if alice_basis == 1:  # X-basis
        qml.Hadamard(wires=0)  # Rotate to X-basis
    
    # Bob's measurement basis
    if bob_basis == 1:  # X-basis measurement
        qml.Hadamard(wires=0)
    
    return qml.sample(qml.PauliZ(0))

def run_bb84(n_bits, eve_present=False, eve_intercept_prob=1.0):
    """Run the full BB84 protocol."""
    # Step 1: Alice generates random bits and bases
    alice_bits = [random.randint(0, 1) for _ in range(n_bits)]
    alice_bases = [random.randint(0, 1) for _ in range(n_bits)]  # 0=Z, 1=X
    
    # Step 2: Bob chooses random measurement bases
    bob_bases = [random.randint(0, 1) for _ in range(n_bits)]
    
    # Step 3: Quantum transmission and measurement
    bob_results = []
    eve_detected_bits = []
    
    for i in range(n_bits):
        if eve_present and random.random() < eve_intercept_prob:
            # Eve intercepts: measures in random basis
            eve_basis = random.randint(0, 1)
            eve_result = bb84_prepare_and_measure(alice_bits[i], alice_bases[i], eve_basis)
            eve_bit = 0 if eve_result == 1 else 1
            eve_detected_bits.append(eve_bit)
            # Eve resends based on her measurement
            bob_result = bb84_prepare_and_measure(eve_bit, eve_basis, bob_bases[i])
        else:
            # No eavesdropping: direct transmission
            bob_result = bb84_prepare_and_measure(alice_bits[i], alice_bases[i], bob_bases[i])
        
        bob_bit = 0 if bob_result == 1 else 1
        bob_results.append(bob_bit)
    
    # Step 4: Basis reconciliation (public channel)
    matching_indices = [i for i in range(n_bits) if alice_bases[i] == bob_bases[i]]
    
    # Step 5: Key sifting
    alice_key = [alice_bits[i] for i in matching_indices]
    bob_key = [bob_results[i] for i in matching_indices]
    
    # Step 6: Error estimation
    errors = sum(a != b for a, b in zip(alice_key, bob_key))
    qber = errors / len(alice_key) if alice_key else 0
    
    return {
        'alice_bits': alice_bits, 'alice_bases': alice_bases,
        'bob_bases': bob_bases, 'bob_results': bob_results,
        'matching_indices': matching_indices,
        'alice_key': alice_key, 'bob_key': bob_key,
        'qber': qber, 'errors': errors
    }

# Run BB84 without eavesdropper
n_bits = 100
result_safe = run_bb84(n_bits, eve_present=False)

print("=" * 60)
print("BB84 Protocol - No Eavesdropper")
print("=" * 60)
print(f"Total qubits sent: {n_bits}")
print(f"Matching bases: {len(result_safe['matching_indices'])} ({100*len(result_safe['matching_indices'])/n_bits:.1f}%)")
print(f"Sifted key length: {len(result_safe['alice_key'])}")
print(f"Errors in sifted key: {result_safe['errors']}")
print(f"QBER: {result_safe['qber']:.4f}")
print(f"\nAlice's key (first 20): {''.join(map(str, result_safe['alice_key'][:20]))}")
print(f"Bob's key   (first 20): {''.join(map(str, result_safe['bob_key'][:20]))}")
print(f"Keys match: {result_safe['alice_key'] == result_safe['bob_key']}")

In [ ]:
# Visualize the BB84 Protocol Flow

def visualize_bb84(result, n_show=20):
    """Visualize BB84 protocol details."""
    fig, ax = plt.subplots(figsize=(16, 8))
    
    n = min(n_show, len(result['alice_bits']))
    basis_names = {0: 'Z', 1: 'X'}
    
    rows = ['Alice Bit', 'Alice Basis', 'Bob Basis', 'Bob Result', 'Match?', 'Key Bit']
    y_positions = [5, 4, 3, 2, 1, 0]
    
    for i in range(n):
        x = i + 0.5
        match = result['alice_bases'][i] == result['bob_bases'][i]
        bg_color = '#e8f5e9' if match else '#ffebee'
        
        ax.add_patch(plt.Rectangle((i, -0.5), 1, 6.5, facecolor=bg_color, edgecolor='gray', alpha=0.5))
        
        ax.text(x, 5, str(result['alice_bits'][i]), ha='center', va='center', fontsize=10, fontweight='bold')
        ax.text(x, 4, basis_names[result['alice_bases'][i]], ha='center', va='center', fontsize=10,
                color='blue' if result['alice_bases'][i]==0 else 'red')
        ax.text(x, 3, basis_names[result['bob_bases'][i]], ha='center', va='center', fontsize=10,
                color='blue' if result['bob_bases'][i]==0 else 'red')
        ax.text(x, 2, str(result['bob_results'][i]), ha='center', va='center', fontsize=10, fontweight='bold')
        ax.text(x, 1, 'Y' if match else 'N', ha='center', va='center', fontsize=10,
                color='green' if match else 'red', fontweight='bold')
        
        if match:
            key_correct = result['alice_bits'][i] == result['bob_results'][i]
            ax.text(x, 0, str(result['alice_bits'][i]), ha='center', va='center', fontsize=10,
                    fontweight='bold', color='green' if key_correct else 'red')
        else:
            ax.text(x, 0, '-', ha='center', va='center', fontsize=10, color='gray')
    
    for i, row in enumerate(rows):
        ax.text(-0.5, y_positions[i], row, ha='right', va='center', fontsize=11, fontweight='bold')
    
    ax.set_xlim(-2, n)
    ax.set_ylim(-1, 6.5)
    ax.set_title('BB84 Protocol Visualization (Green = Matching Bases)', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

visualize_bb84(result_safe)

## 3. Eavesdropper Detection

When Eve intercepts qubits, she must measure them (collapsing the state) and resend. If she measures in the **wrong basis**, she introduces errors:

- Probability Eve chooses correct basis: $1/2$
- When wrong basis: she sends a random state, causing 50% error for Bob
- **Expected QBER with Eve**: $\frac{1}{2} \times \frac{1}{2} = \frac{1}{4} = 25\%$

### Detection Probability

If Alice and Bob check $n$ bits, the probability of detecting Eve:

$$P_{\text{detect}} = 1 - \left(\frac{3}{4}\right)^n$$

In [ ]:
# Eavesdropper Detection: BB84 with Eve

result_eve = run_bb84(n_bits=200, eve_present=True)

print("=" * 60)
print("BB84 Protocol - WITH Eavesdropper (Eve)")
print("=" * 60)
print(f"Total qubits sent: 200")
print(f"Sifted key length: {len(result_eve['alice_key'])}")
print(f"Errors: {result_eve['errors']}")
print(f"QBER: {result_eve['qber']:.4f} (expected ~0.25 with Eve)")
print(f"\nEavesdropper detected: {'YES' if result_eve['qber'] > 0.11 else 'NO'}")
print(f"(Threshold: QBER > 11% indicates eavesdropping)")

# Run multiple trials to show QBER distribution
n_trials = 50
qber_no_eve = []
qber_with_eve = []

for _ in range(n_trials):
    r1 = run_bb84(200, eve_present=False)
    r2 = run_bb84(200, eve_present=True)
    qber_no_eve.append(r1['qber'])
    qber_with_eve.append(r2['qber'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# QBER Distribution
axes[0].hist(qber_no_eve, bins=15, alpha=0.7, color='green', label='No Eve', density=True)
axes[0].hist(qber_with_eve, bins=15, alpha=0.7, color='red', label='With Eve', density=True)
axes[0].axvline(x=0.11, color='black', linestyle='--', linewidth=2, label='Threshold (11%)')
axes[0].set_xlabel('QBER', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('QBER Distribution: With vs Without Eve', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Detection probability vs number of check bits
check_bits = range(1, 51)
detection_prob = [1 - (3/4)**n for n in check_bits]
axes[1].plot(check_bits, detection_prob, 'r-', linewidth=2)
axes[1].axhline(y=0.99, color='green', linestyle='--', alpha=0.7, label='99% detection')
axes[1].axhline(y=0.999, color='blue', linestyle='--', alpha=0.7, label='99.9% detection')
n_99 = np.ceil(np.log(0.01) / np.log(0.75))
axes[1].axvline(x=n_99, color='green', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Number of Check Bits (n)', fontsize=12)
axes[1].set_ylabel('$P_{detect} = 1 - (3/4)^n$', fontsize=12)
axes[1].set_title('Eavesdropper Detection Probability', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].annotate(f'n={int(n_99)} for 99%', xy=(n_99, 0.99), xytext=(n_99+5, 0.93),
                arrowprops=dict(arrowstyle='->', color='green'), fontsize=10, color='green')

plt.tight_layout()
plt.show()

print(f"\nMean QBER (no Eve): {np.mean(qber_no_eve):.4f}")
print(f"Mean QBER (with Eve): {np.mean(qber_with_eve):.4f}")
print(f"Bits needed for 99% detection: {int(n_99)}")
print(f"Bits needed for 99.9% detection: {int(np.ceil(np.log(0.001)/np.log(0.75)))}")

## 4. Information Reconciliation and Privacy Amplification

After key sifting, Alice and Bob perform:

### Error Correction (Cascade Protocol)
Compare parity of random subsets to find and correct errors.

### Privacy Amplification
Apply a universal hash function to shorten the key, removing any information Eve may have:

$$\ell_{\text{final}} = \ell_{\text{sifted}} \times (1 - h(\text{QBER}))$$

where $h(p) = -p\log_2 p - (1-p)\log_2(1-p)$ is the binary entropy function.

In [ ]:
# Information Reconciliation & Privacy Amplification

def binary_entropy(p):
    """Binary entropy function."""
    if p == 0 or p == 1:
        return 0
    return -p * np.log2(p) - (1-p) * np.log2(1-p)

def parity_check_reconciliation(alice_key, bob_key, block_size=4):
    """Simple parity-based error correction."""
    corrected_key = list(bob_key)
    corrections = 0
    
    for i in range(0, len(alice_key) - block_size + 1, block_size):
        alice_parity = sum(alice_key[i:i+block_size]) % 2
        bob_parity = sum(corrected_key[i:i+block_size]) % 2
        
        if alice_parity != bob_parity:
            # Binary search for error within block
            for j in range(i, min(i+block_size, len(alice_key))):
                if alice_key[j] != corrected_key[j]:
                    corrected_key[j] = alice_key[j]
                    corrections += 1
                    break
    
    return corrected_key, corrections

def privacy_amplification(key, target_length):
    """Simple privacy amplification via XOR hashing."""
    amplified = []
    for i in range(target_length):
        # XOR random subset of key bits
        indices = random.sample(range(len(key)), min(3, len(key)))
        bit = 0
        for idx in indices:
            bit ^= key[idx]
        amplified.append(bit)
    return amplified

# Demonstrate on a noisy key (simulate small QBER)
result = run_bb84(500, eve_present=False)
alice_key = result['alice_key']
bob_key = result['bob_key']

print(f"Sifted key length: {len(alice_key)}")
print(f"Initial errors: {result['errors']}")
print(f"Initial QBER: {result['qber']:.4f}")

# Error correction
corrected_key, n_corrections = parity_check_reconciliation(alice_key, bob_key)
remaining_errors = sum(a != b for a, b in zip(alice_key, corrected_key))
print(f"\nAfter error correction:")
print(f"  Corrections made: {n_corrections}")
print(f"  Remaining errors: {remaining_errors}")

# Privacy amplification
qber = result['qber']
if qber > 0:
    final_length = int(len(alice_key) * (1 - binary_entropy(min(qber, 0.49))))
else:
    final_length = len(alice_key)
final_key = privacy_amplification(alice_key, max(final_length, 1))
print(f"\nAfter privacy amplification:")
print(f"  Final key length: {len(final_key)} (from {len(alice_key)})")
print(f"  Key rate: {len(final_key)/500:.3f} bits per qubit sent")

# Visualize key rate vs QBER
qber_range = np.linspace(0, 0.5, 100)
key_rates = [max(0, 1 - 2*binary_entropy(q)) for q in qber_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(qber_range, key_rates, 'b-', linewidth=2)
ax.axvline(x=0.11, color='red', linestyle='--', label='Security threshold (11%)')
ax.fill_between(qber_range, key_rates, alpha=0.1, color='blue')
ax.set_xlabel('QBER', fontsize=12)
ax.set_ylabel('Secret Key Rate', fontsize=12)
ax.set_title('BB84 Secret Key Rate vs QBER', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 5. Security Analysis

### Key Rate vs Distance

In fiber-optic QKD, photon loss increases with distance. The key rate scales as:

$$R \propto \eta_{\text{channel}} = 10^{-\alpha L / 10}$$

where $\alpha \approx 0.2$ dB/km is the fiber attenuation and $L$ is the distance.

In [ ]:
# Security Analysis: Key Rate vs Distance and Partial Interception

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Key rate vs distance (fiber optic)
distances = np.linspace(0, 300, 100)  # km
alpha_db = 0.2  # dB/km for standard fiber
eta = 10**(-alpha_db * distances / 10)
detector_eff = 0.1  # 10% detector efficiency
key_rate_distance = eta * detector_eff * 0.5  # 50% basis matching

axes[0].semilogy(distances, key_rate_distance, 'b-', linewidth=2)
axes[0].set_xlabel('Distance (km)', fontsize=12)
axes[0].set_ylabel('Key Rate (bits/pulse)', fontsize=12)
axes[0].set_title('BB84 Key Rate vs Distance', fontsize=13)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=1e-6, color='red', linestyle='--', alpha=0.7, label='Practical limit')
axes[0].legend(fontsize=10)

# 2. QBER vs Eve's interception fraction
intercept_fracs = np.linspace(0, 1, 50)
expected_qber = intercept_fracs * 0.25  # Eve causes 25% error on intercepted bits

# Simulate
simulated_qber = []
for frac in [0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    qbers = [run_bb84(200, eve_present=True if frac > 0 else False, 
                       eve_intercept_prob=frac)['qber'] for _ in range(20)]
    simulated_qber.append((frac, np.mean(qbers), np.std(qbers)))

sim_fracs = [s[0] for s in simulated_qber]
sim_means = [s[1] for s in simulated_qber]
sim_stds = [s[2] for s in simulated_qber]

axes[1].plot(intercept_fracs, expected_qber, 'r-', linewidth=2, label='Theoretical')
axes[1].errorbar(sim_fracs, sim_means, yerr=sim_stds, fmt='bo', markersize=6, label='Simulated')
axes[1].axhline(y=0.11, color='green', linestyle='--', linewidth=1.5, label='Security threshold')
axes[1].set_xlabel('Eve Interception Fraction', fontsize=12)
axes[1].set_ylabel('QBER', fontsize=12)
axes[1].set_title('QBER vs Eavesdropping Intensity', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# 3. Key length vs number of qubits sent
n_qubits_sent = [50, 100, 200, 500, 1000]
safe_key_lengths = []
eve_key_lengths = []

for n in n_qubits_sent:
    safe_results = [len(run_bb84(n, eve_present=False)['alice_key']) for _ in range(10)]
    safe_key_lengths.append(np.mean(safe_results))

axes[2].plot(n_qubits_sent, safe_key_lengths, 'go-', linewidth=2, markersize=8, label='Sifted key length')
axes[2].plot(n_qubits_sent, [n*0.5 for n in n_qubits_sent], 'b--', linewidth=1.5, label='Theoretical (N/2)')
axes[2].set_xlabel('Qubits Sent', fontsize=12)
axes[2].set_ylabel('Key Length (bits)', fontsize=12)
axes[2].set_title('Key Generation Efficiency', fontsize=13)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. B92 Protocol: A Simpler Alternative

The **B92 protocol** (Bennett, 1992) uses only **two non-orthogonal states**:

- Bit 0: $|0\rangle$
- Bit 1: $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$

Bob measures in either the Z or X basis. He only gets a conclusive result when his basis is **complementary** to Alice's encoding.

In [ ]:
# B92 Protocol Implementation

@qml.qnode(dev)
def b92_prepare_and_measure(alice_bit, bob_basis):
    """B92: Alice sends |0> for bit 0, |+> for bit 1."""
    if alice_bit == 1:
        qml.Hadamard(wires=0)  # |+> for bit 1
    if bob_basis == 1:  # X-basis measurement
        qml.Hadamard(wires=0)
    return qml.sample(qml.PauliZ(0))

def run_b92(n_bits, eve_present=False):
    """Run B92 protocol."""
    alice_bits = [random.randint(0, 1) for _ in range(n_bits)]
    bob_bases = [random.randint(0, 1) for _ in range(n_bits)]
    
    alice_key = []
    bob_key = []
    
    for i in range(n_bits):
        if eve_present:
            eve_basis = random.randint(0, 1)
            eve_result = b92_prepare_and_measure(alice_bits[i], eve_basis)
            eve_bit = 0 if eve_result == 1 else 1
            result = b92_prepare_and_measure(eve_bit, bob_bases[i])
        else:
            result = b92_prepare_and_measure(alice_bits[i], bob_bases[i])
        
        bob_bit = 0 if result == 1 else 1
        
        # B92: Bob keeps only conclusive results
        # Conclusive when Bob gets |1> in Z-basis or |-> in X-basis
        if bob_bit == 1:  # Conclusive result
            alice_key.append(alice_bits[i])
            bob_key.append(bob_bases[i])  # Bob's key bit = his basis choice
    
    errors = sum(a != b for a, b in zip(alice_key, bob_key))
    qber = errors / len(alice_key) if alice_key else 0
    
    return {'alice_key': alice_key, 'bob_key': bob_key, 'qber': qber,
            'efficiency': len(alice_key) / n_bits}

# Compare BB84 and B92
n_compare = 500
n_trials = 30

bb84_eff, b92_eff = [], []
bb84_qber_eve, b92_qber_eve = [], []

for _ in range(n_trials):
    r_bb84 = run_bb84(n_compare, eve_present=False)
    r_b92 = run_b92(n_compare, eve_present=False)
    bb84_eff.append(len(r_bb84['alice_key']) / n_compare)
    b92_eff.append(r_b92['efficiency'])
    
    r_bb84_eve = run_bb84(n_compare, eve_present=True)
    r_b92_eve = run_b92(n_compare, eve_present=True)
    bb84_qber_eve.append(r_bb84_eve['qber'])
    b92_qber_eve.append(r_b92_eve['qber'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Efficiency comparison
protocols = ['BB84', 'B92']
efficiencies = [np.mean(bb84_eff), np.mean(b92_eff)]
eff_stds = [np.std(bb84_eff), np.std(b92_eff)]
colors = ['steelblue', 'coral']
axes[0].bar(protocols, efficiencies, yerr=eff_stds, color=colors, alpha=0.8, capsize=5)
axes[0].set_ylabel('Key Generation Efficiency', fontsize=12)
axes[0].set_title('Protocol Efficiency (Key bits / Qubits sent)', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')

# QBER with Eve
axes[1].boxplot([bb84_qber_eve, b92_qber_eve], labels=['BB84', 'B92'])
axes[1].axhline(y=0.25, color='red', linestyle='--', alpha=0.7, label='Expected BB84 QBER')
axes[1].set_ylabel('QBER (with Eve)', fontsize=12)
axes[1].set_title('Eavesdropper-Induced Error Rate', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"BB84 efficiency: {np.mean(bb84_eff):.3f} | B92 efficiency: {np.mean(b92_eff):.3f}")
print(f"BB84 QBER (Eve): {np.mean(bb84_qber_eve):.3f} | B92 QBER (Eve): {np.mean(b92_qber_eve):.3f}")

## 7. Conclusion

### Key Results

| Metric | BB84 | B92 |
|--------|------|-----|
| **States used** | 4 (two bases) | 2 (non-orthogonal) |
| **Key efficiency** | ~50% | ~25% |
| **QBER with Eve** | ~25% | Variable |
| **Implementation** | Standard | Simpler |

### Real-World QKD Systems

| System | Organization | Distance | Key Rate |
|--------|-------------|----------|----------|
| **Micius Satellite** | Chinese Academy of Sciences | 1,200 km | ~1 kbps |
| **Tokyo QKD Network** | NICT Japan | Metro scale | ~100 kbps |
| **Cerberis** | ID Quantique | 100 km fiber | ~1-10 kbps |
| **Twin-Field QKD** | Various | 600+ km fiber | ~0.1 bps |

### Key Takeaways

1. **BB84 provides information-theoretic security** — unbreakable even with quantum computers
2. **Eavesdropping is always detectable** — quantum mechanics ensures Eve disturbs the channel
3. **Real-world deployment is happening** — satellite QKD, metro fiber networks, commercial products
4. **Post-quantum cryptography vs QKD**: PQC is a software solution; QKD is a physics solution — both have roles

### References

1. Bennett, C.H. & Brassard, G. "Quantum cryptography: Public key distribution and coin tossing." *Proc. IEEE Int. Conf. Computers, Systems and Signal Processing* (1984).
2. Bennett, C.H. "Quantum cryptography using any two nonorthogonal states." *Phys. Rev. Lett.* 68.21 (1992): 3121.
3. Gisin, N., et al. "Quantum cryptography." *Rev. Mod. Phys.* 74.1 (2002): 145.
4. Liao, S.K., et al. "Satellite-to-ground quantum key distribution." *Nature* 549 (2017): 43-47.